# Trotter-Suzuki Time Evolution

Evolve a state under a ZZ + transverse field Hamiltonian using first-order Trotter decomposition, compared against exact matrix-exponential evolution.

In [ ]:
import numpy as np
import qiskit as qk
from qiskit.quantum_info import Operator, SparsePauliOp, Statevector

## Hamiltonian: H = J·Z₀Z₁ + h·(X₀ + X₁)

In [ ]:
J, H_FIELD = 1.0, 0.5
HAMILTONIAN = SparsePauliOp.from_list([
    ("ZZ", complex(J)),
    ("XI", complex(H_FIELD)),
    ("IX", complex(H_FIELD)),
])
print(f"H = {J:.1f}·Z₀Z₁ + {H_FIELD:.1f}·(X₀ + X₁)")

## Exact vs Trotter evolution

In [ ]:
def exact_evolution_operator(t):
    mat = HAMILTONIAN.to_matrix()
    eigenvalues, eigenvectors = np.linalg.eigh(mat)
    exp_diag = np.exp(-1j * eigenvalues * t)
    U = eigenvectors @ np.diag(exp_diag) @ eigenvectors.conj().T
    return Operator(U)

def trotter_step(dt):
    qc_zz = qk.QuantumCircuit(2)
    qc_zz.cx(0, 1)
    qc_zz.rz(2.0 * J * dt, 1)
    qc_zz.cx(0, 1)
    qc_x = qk.QuantumCircuit(2)
    qc_x.rx(2.0 * H_FIELD * dt, 0)
    qc_x.rx(2.0 * H_FIELD * dt, 1)
    return Operator(qc_zz.compose(qc_x))

def trotter_evolution(state, t, n_steps):
    dt = t / n_steps
    U_step = trotter_step(dt)
    evolved = state
    for _ in range(n_steps):
        evolved = evolved.evolve(U_step)
    return evolved

In [ ]:
init = Statevector.from_int(1, dims=4)  # |01⟩
times = [0.5, 1.0, 2.0, 5.0]
n_trotter_steps = 20

for t in times:
    exact_state = init.evolve(exact_evolution_operator(t))
    exact_probs = exact_state.probabilities_dict()
    trotter_state = trotter_evolution(init, t, n_trotter_steps)
    trotter_probs = trotter_state.probabilities_dict()
    fidelity = float(np.abs(np.dot(exact_state.data.conj(), trotter_state.data)) ** 2)
    print(f"t={t:.1f}  exact: {exact_probs}  Trotter: {trotter_probs}  fidelity: {fidelity:.6f}")

## Trotter convergence

In [ ]:
t_fixed = 2.0
exact_final = init.evolve(exact_evolution_operator(t_fixed))
for n_steps in [1, 2, 5, 10, 20, 50]:
    trotter_final = trotter_evolution(init, t_fixed, n_steps)
    fidelity = float(np.abs(np.dot(exact_final.data.conj(), trotter_final.data)) ** 2)
    dist = float(np.linalg.norm(exact_final.data - trotter_final.data))
    print(f"  steps={n_steps:>3d}  fidelity={fidelity:.6f}  dist={dist:.8f}")